# S06 — Feature comparison (Contribution 2)

Cross-border inputs **on** vs **off** for the consumption-based two-tier model (E3).
`off` drops the `net_flow_*` and `partner_ci_*` columns, so the orchestrator builds a
source-only Tier 2 (no flow ANN, no partner-CI channel). Each cell is seed-averaged
(best-by-validation) on the real split: train 2021-2025, val 2026 Jan-Apr, Test A May 2026.

Runs `scripts/feature_comparison.py`, which **checkpoints after every cell and is resumable**:
if the Colab session drops, just re-run the *Run* cell and it continues. Per cell it saves
predictions (npz), per-seed metrics (csv), and the **full trained E3** (`save_e3`, reloadable
via `load_e3` for Test B). All artifacts go to Drive, so they survive the session.


## 1. Setup (mount Drive, get the repo)


In [ ]:
import os, sys, getpass
IN_COLAB = 'google.colab' in sys.modules
REPO = '/content/carbon-intensity-forecast'
DRIVE = '/content/drive/MyDrive/carbon-intensity-forecast'
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    if not os.path.isdir(REPO):
        tok = getpass.getpass('GitHub token (repo read): ')
        os.system(f'git clone https://{tok}@github.com/nicolaswilches/carbon-intensity-forecast.git {REPO}')
    os.system(f'cd {REPO} && git pull -q')
    DATA_ROOT = f'{DRIVE}/data/processed'
    OUTDIR    = f'{DRIVE}/outputs_feature_comparison'
else:
    REPO = os.path.abspath('..')
    DATA_ROOT = os.path.join(REPO, 'data', 'processed')
    OUTDIR    = os.path.join(REPO, 'outputs')
sys.path.insert(0, os.path.join(REPO, 'src'))   # carbon_forecast importable; TF/keras/etc. ship with Colab
os.environ['KERAS_BACKEND'] = 'tensorflow'
print('repo :', REPO)
print('data :', DATA_ROOT)
print('out  :', OUTDIR)

## 2. Verify the 5 processed parquets are on Drive


In [ ]:
import glob
files = sorted(glob.glob(os.path.join(DATA_ROOT, '*.parquet')))
print(len(files), 'parquet files:', [os.path.basename(f) for f in files])
assert len(files) >= 5, 'Upload data/processed/*.parquet to Drive at ' + DATA_ROOT

## 3. Run (full config, several hours on GPU; resumable)

`MODES=off,on` runs both conditions so the **flow-on** models are also saved (the main
consumption model, reusable for the Test B head-to-head). Drop to `MODES=off` to only run
the ablation and reuse the existing on-baseline. Re-run this cell after any disconnect.


In [ ]:
env = dict(DATA_ROOT=DATA_ROOT, OUTDIR=OUTDIR, MODES='off,on', SEEDS='0,1,2', KERAS_BACKEND='tensorflow')
envstr = ' '.join(f'{k}={v}' for k, v in env.items())
!cd {REPO} && {envstr} python scripts/feature_comparison.py

## 4. Results: per-zone off vs on


In [ ]:
import pandas as pd
df = pd.read_csv(os.path.join(OUTDIR, 'feature_comparison.csv'))
piv = df.pivot(index='zone', columns='flow', values='test_mape')
piv['flows_help'] = piv['on'] < piv['off']
piv